# Week 2

In [101]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

df = pd.read_csv("../Dataset/MQTTEEB-D_cleaned_data.csv", low_memory=False)
print(df.isna().sum())

timestamp                   0
tcp_flags                   0
tcp_time_delta              0
tcp_len                     0
mqtt_conack_flags           0
mqtt_conflag_cleansess      0
mqtt_conflags               0
mqtt_dupflag                0
mqtt_hdrflags               0
mqtt_kalive                 0
mqtt_msg                    0
mqtt_qos                    0
label                     246
dtype: int64


Some rows have null value in "label" column.

So we will remove then, 246/222813 is not a big deal.

In [102]:
df = df.dropna()

We don't need the time

In [103]:
df_bis = df[["timestamp", "tcp_time_delta"]]
df = df.drop(columns = ["timestamp", "tcp_time_delta"])

In [104]:
# mqtt_conack_flags --> Octet du variable header packet CONNACK: réponse du broker à un CONNACK 
# mqtt_conflags --> Octet du variable header packet CONNECT: réponse envoyé par un client lors d'un CONNECT
# mqtt_hdrflags --> Octet du Fixed header : CONNECT PUBLISH OU DISCONNECT

mqtt_hex_cols = ["mqtt_conack_flags", "mqtt_conflags", "mqtt_hdrflags", "tcp_flags"]

for col in mqtt_hex_cols:
    df[f"{col}_int"] = [int(x, 16) for x in df[col]]

df["mqtt_msg_type"] = df["mqtt_hdrflags_int"] // 16

mqtt_packet_types = {
    1: "CONNECT", 2: "CONNACK", 3: "PUBLISH", 4: "PUBACK",
    8: "SUBSCRIBE", 9: "SUBACK", 12: "PINGREQ", 13: "PINGRESP", 14: "DISCONNECT"
}

df_bis["mqtt_msg_type_str"] = df["mqtt_msg_type"].map(mqtt_packet_types).fillna("OTHER")

tcp_flags_strings_list = []
for val in df["tcp_flags_int"]:
    tcp_flags_strings = ""

    if val >= 128: 
        tcp_flags_strings += "CWR "
        val -= 128
    if val >= 64: 
        tcp_flags_strings += "ECE "
        val -= 64
    if val >= 32: 
        tcp_flags_strings += "URG "
        val -= 32
    if val >= 16: 
        tcp_flags_strings += "ACK "
        val -= 16
    if val >= 8: 
        tcp_flags_strings += "PSH "
        val -= 8
    if val >= 4: 
        tcp_flags_strings += "RST "
        val -= 4
    if val >= 2: 
        tcp_flags_strings += "SYN "
        val -= 2
    if val == 1: 
        tcp_flags_strings += "FIN "
        val -= 1
    tcp_flags_strings_list.append(tcp_flags_strings)

conflags_strings_list = []
for val in df["mqtt_conflags_int"]:
    flags = ""
    if val >= 128:
        flags += "USER "
        val -= 128
    if val >= 64:
        flags += "PASS "
        val -= 64
    if val >= 32:
        flags += "WILL_RET "
        val -= 32
    if val >= 16:
        val -= 16  # Bit QoS haut
    if val >= 8:
        val -= 8   # Bit QoS bas
    if val >= 4:
        flags += "WILL_FLAG "
        val -= 4
    if val >= 2:
        flags += "CLEAN_SESS "
        val -= 2
    conflags_strings_list.append(flags.strip() if flags else "NONE")

df_bis["mqtt_conflags_strings"] = conflags_strings_list

df_bis["tcp_flags_strings"] = tcp_flags_strings_list


# mqtt_hdrflags --> 
#
#
    

In [105]:
df.head(100)


,tcp_flags,tcp_len,mqtt_conack_flags,mqtt_conflag_cleansess,mqtt_conflags,mqtt_dupflag,mqtt_hdrflags,mqtt_kalive,mqtt_msg,mqtt_qos,label,mqtt_conack_flags_int,mqtt_conflags_int,mqtt_hdrflags_int,tcp_flags_int,mqtt_msg_type
0,0x0018,617,0,0,0,0,0x35,0,37372,2,legitimate,0,0,53,24,3
1,0x0018,609,0,0,0,0,0x35,0,37373,2,legitimate,0,0,53,24,3
2,0x0018,614,0,0,0,0,0x35,0,37374,2,legitimate,0,0,53,24,3
3,0x0018,615,0,0,0,0,0x35,0,37375,2,legitimate,0,0,53,24,3
4,0x0018,615,0,0,0,0,0x35,0,37376,2,legitimate,0,0,53,24,3
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,0x0018,40,0,1,0x82,0,0x10,60,0,0,dos,0,130,16,24,1
96,0x0018,40,0,1,0x82,0,0x10,60,0,0,dos,0,130,16,24,1
97,0x0018,40,0,1,0x82,0,0x10,60,0,0,dos,0,130,16,24,1
98,0x0018,40,0,1,0x82,0,0x10,60,0,0,dos,0,130,16,24,1


In [106]:
df_bis.head(100)


,timestamp,tcp_time_delta,mqtt_msg_type_str,mqtt_conflags_strings,tcp_flags_strings
0,1.725852e+09,1.725852e+09,PUBLISH,NONE,ACK PSH
1,1.725852e+09,1.725852e+09,PUBLISH,NONE,ACK PSH
2,1.725852e+09,1.725852e+09,PUBLISH,NONE,ACK PSH
3,1.725852e+09,1.725852e+09,PUBLISH,NONE,ACK PSH
4,1.725852e+09,1.725852e+09,PUBLISH,NONE,ACK PSH
...,...,...,...,...,...
95,1.725852e+09,1.725852e+09,CONNECT,USER CLEAN_SESS,ACK PSH
96,1.725852e+09,1.725852e+09,CONNECT,USER CLEAN_SESS,ACK PSH
97,1.725852e+09,1.725852e+09,CONNECT,USER CLEAN_SESS,ACK PSH
98,1.725852e+09,1.725852e+09,CONNECT,USER CLEAN_SESS,ACK PSH


3 autres à faire !!

In [107]:
target = []
for l in df["label"]:
    if l != "legitimate":
        target.append(1)
    else: 
        target.append(0)

df["target"] = target
df_bis["label"] = df["label"]
df = df.drop(columns = ["label"])



In [108]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 222567 entries, 0 to 222812
Data columns (total 16 columns):
 #   Column                  Non-Null Count   Dtype 
---  ------                  --------------   ----- 
 0   tcp_flags               222567 non-null  object
 1   tcp_len                 222567 non-null  int64 
 2   mqtt_conack_flags       222567 non-null  object
 3   mqtt_conflag_cleansess  222567 non-null  int64 
 4   mqtt_conflags           222567 non-null  object
 5   mqtt_dupflag            222567 non-null  int64 
 6   mqtt_hdrflags           222567 non-null  object
 7   mqtt_kalive             222567 non-null  int64 
 8   mqtt_msg                222567 non-null  int64 
 9   mqtt_qos                222567 non-null  int64 
 10  mqtt_conack_flags_int   222567 non-null  int64 
 11  mqtt_conflags_int       222567 non-null  int64 
 12  mqtt_hdrflags_int       222567 non-null  int64 
 13  tcp_flags_int           222567 non-null  int64 
 14  mqtt_msg_type           222567 non-null  

In [109]:
X = df.drop(columns=["target"])
y = df["target"]

X_train, X_test, y_train, y_test = train_test_split(X, y, train_size=0.75, stratify=y, random_state=42)



In [110]:
# X